## Custom Tools Method 1 - > using @tool

### Step 1: Import the tool decorator


In [8]:
from langchain_core.tools import tool

### Step 2: Define the custom tool


In [9]:
@tool #decorator to register the function as a tool
def multiply_numbers(num1: float, num2: float) -> float: # giving type hints for the function's arguments and return type
    """
    Multiplies two numbers and returns the result.
    
    Args:
        num1 (float): The first number.
        num2 (float): The second number.
        
    Returns:
        float: The product of the two numbers.
    """
    return num1 * num2


### Step 3: Invoke the custom tool


In [12]:
result=multiply_numbers.invoke({"num1": 3, "num2": 4})  # This will return 12

### Step 4: Print the result


In [14]:
print(result)

12.0


### Step 5: Check tool metadata and schema


In [15]:
print(multiply_numbers.name)  # Output: multiply_numbers
print(multiply_numbers.description)  # Output: Multiplies two numbers and returns the result.
print(multiply_numbers.args)  # Output: {'num1': 'float', 'num2': 'float'}

multiply_numbers
Multiplies two numbers and returns the result.

Args:
    num1 (float): The first number.
    num2 (float): The second number.

Returns:
    float: The product of the two numbers.
{'num1': {'title': 'Num1', 'type': 'number'}, 'num2': {'title': 'Num2', 'type': 'number'}}


In [17]:
multiply_numbers.args_schema.model_json_schema()  # This will return the JSON schema of the tool's arguments and return type

{'description': 'Multiplies two numbers and returns the result.\n\nArgs:\n    num1 (float): The first number.\n    num2 (float): The second number.\n\nReturns:\n    float: The product of the two numbers.',
 'properties': {'num1': {'title': 'Num1', 'type': 'number'},
  'num2': {'title': 'Num2', 'type': 'number'}},
 'required': ['num1', 'num2'],
 'title': 'multiply_numbers',
 'type': 'object'}

### Methosd 2 using StructedTool and Pydantic

### Step 1: Import StructuredTool and Pydantic


In [20]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

### Step 2: Define the argument schema


In [21]:
class MultiplyNumbersArgs(BaseModel):
    num1: float = Field(..., description="The first number.")
    num2: float = Field(..., description="The second number.")

### Step 3: Create the StructuredTool


In [22]:
def multiplying(num1: float, num2: float) -> float:
    return num1 * num2

### Step 4: Invoke the tool and print output


In [23]:
multiply_tool = StructuredTool(
    name="multiply_numbers",
    description="Multiplies two numbers and returns the result.",
    args_schema=MultiplyNumbersArgs,
    func=multiplying
)

### Step 5: Inspect metadata and schema


In [24]:
results_st = multiply_tool.invoke({"num1":2,"num2":3})
print(results_st)

6.0


In [26]:
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

multiply_numbers
Multiplies two numbers and returns the result.
{'num1': {'description': 'The first number.', 'title': 'Num1', 'type': 'number'}, 'num2': {'description': 'The second number.', 'title': 'Num2', 'type': 'number'}}


In [28]:
multiply_tool.args_schema.model_json_schema()

{'properties': {'num1': {'description': 'The first number.',
   'title': 'Num1',
   'type': 'number'},
  'num2': {'description': 'The second number.',
   'title': 'Num2',
   'type': 'number'}},
 'required': ['num1', 'num2'],
 'title': 'MultiplyNumbersArgs',
 'type': 'object'}

# Using Method 3 BaseTool

### Step 1: Import BaseTool and typing support


In [29]:
from langchain_core.tools import BaseTool
from typing import Type

### Step 2: Define the BaseTool input schema


In [30]:
class MultiplyInput(BaseModel):
    a : int = Field(required = True, description = "First number")
    b : int = Field(required = True, description = "Second number")

/var/folders/q1/5c6nynw1077f97t4m6258zy40000gn/T/ipykernel_74237/2991621374.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a : int = Field(required = True, description = "First number")
/var/folders/q1/5c6nynw1077f97t4m6258zy40000gn/T/ipykernel_74237/2991621374.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b : int = Field(required = True, description = "Second number")


### Step 3: Build the BaseTool subclass


In [32]:
class MultiplyingTool(BaseTool):
    name: str ="multiply"
    description: str = "Multiply 2 numbers"
    args_schema: Type[BaseModel] = MultiplyInput
    def _run(self, a:int, b:int) -> float:
        return a*b

### Step 4: Invoke the BaseTool


In [33]:
multiply_tool3 = MultiplyingTool()

### Step 5: Inspect its metadata and schema


In [35]:
result = multiply_tool3.invoke({"a":3,"b":4})

In [36]:
print(result)

12


In [37]:
print(multiply_tool3.name)
print(multiply_tool3.description)
print(multiply_tool3.args)

multiply
Multiply 2 numbers
{'a': {'description': 'First number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'integer'}}


In [38]:
multiply_tool3.args_schema.model_json_schema()

{'properties': {'a': {'description': 'First number',
   'required': True,
   'title': 'A',
   'type': 'integer'},
  'b': {'description': 'Second number',
   'required': True,
   'title': 'B',
   'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'MultiplyInput',
 'type': 'object'}